In [25]:
import pandas as pd
import unicodedata
import re

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 26, Finished, Available, Finished)

In [26]:
def normalize_city(text):
    if pd.isna(text):  # keep NaN safe
        return text
    
    # remove accents/diacritics
    text = ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )
    
    # lowercase + strip spaces
    text = text.lower().strip()
    
    # collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text)
    
    # remove special characters like '-' or '/'and only keep apostrophe
    text = re.sub(r'[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+£]', ' ', text)
    
    return text

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 27, Finished, Available, Finished)

In [27]:
# Create a pandas dataframe for the sellers data set and read the first few records
sellers = pd.read_csv("/lakehouse/default/Files/olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
print(sellers)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 28, Finished, Available, Finished)

                             seller_id seller_zip_code_prefix  \
0     3442f8959a84dea7ee197c632cb2df15                  13023   
1     d1b65fc7debc3361ea86b5f14c68d2e2                  13844   
2     ce3ad9de960102d0677a81f5d0bb7b2d                  20031   
3     c0f3eea2e14555b6faeea3dd58c1b1c3                  04195   
4     51a04a8a6bdcb23deccc82b0b80742cf                  12914   
...                                ...                    ...   
3090  98dddbc4601dd4443ca174359b237166                  87111   
3091  f8201cab383e484733266d1906e2fdfa                  88137   
3092  74871d19219c7d518d0090283e03c137                  04650   
3093  e603cf3fec55f8697c9059638d6c8eb5                  96080   
3094  9e25199f6ef7e7c347120ff175652c3b                  12051   

            seller_city seller_state  
0              campinas           SP  
1            mogi guacu           SP  
2        rio de janeiro           RJ  
3             sao paulo           SP  
4     braganca paulista 

In [28]:
# Convert all text columns to pandas string dtype
sellers = sellers.astype("string", copy=True, errors='raise')

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 29, Finished, Available, Finished)

In [29]:
assert all(sellers.dtypes == "string")

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 30, Finished, Available, Finished)

In [30]:
# Inspect the shape of the dataframe
print(sellers.shape)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 31, Finished, Available, Finished)

(3095, 4)


In [31]:
#Inspect the data types of each column and check for null values
print(sellers.info())

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 32, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   string
 1   seller_zip_code_prefix  3095 non-null   string
 2   seller_city             3095 non-null   string
 3   seller_state            3095 non-null   string
dtypes: string(4)
memory usage: 96.8 KB
None


In [32]:
# Check for values that have anomalies in seller_city

# Look for special characters in the seller city field 
char = r"[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+0£]"
mask = sellers["seller_city"].astype(str).str.contains(char, regex=True)

# Filter the DataFrame
sellers_with_special_chars = sellers[mask]
print(sellers_with_special_chars['seller_city'])
print("Total affected records:", len(sellers_with_special_chars))

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 33, Finished, Available, Finished)

78                                    lages - sc
237                                 auriflama/sp
246                        sao paulo / sao paulo
517                                     04482255
551     novo hamburgo, rio grande do sul, brasil
622                               cariacica / es
826                               sao paulo - sp
869                                       sbc/sp
874               arraial d'ajuda (porto seguro)
945                        santo andre/sao paulo
1004                                     sp / sp
1159                              maua/sao paulo
1337                        mogi das cruzes / sp
1346              rio de janeiro \rio de janeiro
1447                     barbacena/ minas gerais
1580                              sao paulo - sp
1610                                   andira-pr
1649             rio de janeiro / rio de janeiro
1712                                  pinhais/pr
1920                  ribeirao preto / sao paulo
2162                

In [33]:
# Run the normalize function on the seller_city column
sellers['seller_city'] = sellers['seller_city'].apply(normalize_city)
print(sellers.head())

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 34, Finished, Available, Finished)

                          seller_id seller_zip_code_prefix        seller_city  \
0  3442f8959a84dea7ee197c632cb2df15                  13023           campinas   
1  d1b65fc7debc3361ea86b5f14c68d2e2                  13844         mogi guacu   
2  ce3ad9de960102d0677a81f5d0bb7b2d                  20031     rio de janeiro   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                  04195          sao paulo   
4  51a04a8a6bdcb23deccc82b0b80742cf                  12914  braganca paulista   

  seller_state  
0           SP  
1           SP  
2           RJ  
3           SP  
4           SP  


In [34]:
# Run check for values that have anomalies in seller_city

# Look for special characters in the seller city field 
char = r"[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+0£]"
mask = sellers["seller_city"].astype(str).str.contains(char, regex=True)

# Filter the DataFrame
sellers_with_special_chars = sellers[mask]
print(sellers_with_special_chars['seller_city'])
print("Total affected records:", len(sellers_with_special_chars))

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 35, Finished, Available, Finished)

517    04482255
Name: seller_city, dtype: object
Total affected records: 1


In [35]:
# Read city_lookup reference table

city_lookup_by_zipcode = pd.read_parquet("abfss://datasquirrels@onelake.dfs.fabric.microsoft.com/SilverLakehouse.Lakehouse/Tables/dbo/city_lookup_by_zipcode")
print(city_lookup_by_zipcode)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 36, Finished, Available, Finished)

                  city state zip_start zip_end
0          00000-00999  None     00000   00999
1            São Paulo    SP     01000   05999
2               Osasco    SP     06000   06299
3          Carapicuíba    SP     06300   06399
4              Barueri    SP     06400   06499
...                ...   ...       ...     ...
22431          Charrua    RS     99960   99964
22432       Água Santa    RS     99965   99969
22433          Ciríaco    RS     99970   99979
22434  David Canabarro    RS     99980   99989
22435        Muliterno    RS     99990   99999

[22436 rows x 4 columns]


In [36]:
print(city_lookup_by_zipcode.info())

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 37, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22436 entries, 0 to 22435
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   city       22436 non-null  object
 1   state      22418 non-null  object
 2   zip_start  22436 non-null  object
 3   zip_end    22436 non-null  object
dtypes: object(4)
memory usage: 701.3+ KB
None


In [37]:
# Run the normalize function on the city column
city_lookup_by_zipcode['city_sellers_check'] = city_lookup_by_zipcode['city'].apply(normalize_city)
print(city_lookup_by_zipcode)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 38, Finished, Available, Finished)

                  city state zip_start zip_end city_sellers_check
0          00000-00999  None     00000   00999        00000 00999
1            São Paulo    SP     01000   05999          sao paulo
2               Osasco    SP     06000   06299             osasco
3          Carapicuíba    SP     06300   06399        carapicuiba
4              Barueri    SP     06400   06499            barueri
...                ...   ...       ...     ...                ...
22431          Charrua    RS     99960   99964            charrua
22432       Água Santa    RS     99965   99969         agua santa
22433          Ciríaco    RS     99970   99979            ciriaco
22434  David Canabarro    RS     99980   99989    david canabarro
22435        Muliterno    RS     99990   99999          muliterno

[22436 rows x 5 columns]


city_lookup_by_zipcode has columns called zip_start and zip end to signify the range of zipcode prefixes that a single city may have. 

To use this range to lookup the seller_city using the seller_zip_code prefix, we need to convert the zip code columns in both tables to integers to be able to check within the range and print the matching city name from city_lookup_by_zipcode table. 

In [38]:
# Keep original zip as string
sellers["seller_zip_code_prefix_str"] = sellers["seller_zip_code_prefix"]

# Cast to int for matching
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].astype(int)

city_lookup_by_zipcode["zip_start"] = city_lookup_by_zipcode["zip_start"].astype(int)
city_lookup_by_zipcode["zip_end"] = city_lookup_by_zipcode["zip_end"].astype(int)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 39, Finished, Available, Finished)

In [39]:
def find_city(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["city_sellers_check"]  # Return the first matching city
    return None

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 40, Finished, Available, Finished)

In [40]:
def find_state(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["state"]  # Return the first matching state
    return None

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 41, Finished, Available, Finished)

In [41]:
sellers["matched_city"] = sellers["seller_zip_code_prefix"].apply(find_city)
sellers["matched_state"] = sellers["seller_zip_code_prefix"].apply(find_state)
print(sellers)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 42, Finished, Available, Finished)

                             seller_id  seller_zip_code_prefix  \
0     3442f8959a84dea7ee197c632cb2df15                   13023   
1     d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2     ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3     c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4     51a04a8a6bdcb23deccc82b0b80742cf                   12914   
...                                ...                     ...   
3090  98dddbc4601dd4443ca174359b237166                   87111   
3091  f8201cab383e484733266d1906e2fdfa                   88137   
3092  74871d19219c7d518d0090283e03c137                    4650   
3093  e603cf3fec55f8697c9059638d6c8eb5                   96080   
3094  9e25199f6ef7e7c347120ff175652c3b                   12051   

            seller_city seller_state seller_zip_code_prefix_str  \
0              campinas           SP                      13023   
1            mogi guacu           SP                      13844   
2     

In [42]:
# Check the sellers dataframes for rows with any null value
rows_with_nulls = sellers[sellers.isnull().any(axis=1)]
print(rows_with_nulls)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 43, Finished, Available, Finished)

Empty DataFrame
Columns: [seller_id, seller_zip_code_prefix, seller_city, seller_state, seller_zip_code_prefix_str, matched_city, matched_state]
Index: []


In [43]:
# Cross validate seller_city and matched_city
sellers["validate_city_match"] = (sellers["seller_city"] == sellers["matched_city"])

# Cross validate seller_state and matched_state
sellers["validate_state_match"] = (sellers["seller_state"] == sellers["matched_state"])

# Rows where either city or state validation failed
invalid_rows = sellers[(~sellers["validate_city_match"]) | (~sellers["validate_state_match"])]

print(invalid_rows)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 44, Finished, Available, Finished)

                             seller_id  seller_zip_code_prefix  \
29    406822777a0b9eb5c50e442dd4cd3ec5                   18500   
43    5c030029b5916fed0986310385ec9009                   88075   
70    f410c8873029fcc3809b9df6d0b28914                   95076   
78    731ef20c231d9a7103a425e83fd91271                   88501   
103   99cd94252748d2bdde08e17858233602                   12401   
...                                ...                     ...   
3015  7f5e4d5efad7e44b91115dd1decb65f3                   12306   
3026  17e34d8224d27a541263c4c64b11a56b                   14085   
3036  6b9b80d53ba3676eafe60268a810b5a1                   31160   
3044  778323240ce2830d68aab11794e00bfb                   13600   
3071  1d1bbb8ac1581824986f582583fff01d                   89082   

              seller_city seller_state seller_zip_code_prefix_str  \
29                  tatui           SP                      18500   
43               sao jose           SC                      88075   


There are 131 rows where either the city or the state does not match. We'll make the assumption that the zip code prefix provided in the sellers table to be true and update the seller_city and seller_state using the city_lookup_by_zipcode table to ensure consistency in the sellers data. 

We will take the original city name and state name from the city_look_up_by_zipcode for consistency with Brazil's naming convention.

In [44]:
def find_city_lookup(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["city"]  # Return the first matching city
    return None

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 45, Finished, Available, Finished)

In [45]:
def find_state_lookup(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["state"]  # Return the first matching state
    return None

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 46, Finished, Available, Finished)

In [46]:
sellers["lookup_city"] = sellers["seller_zip_code_prefix"].apply(find_city_lookup)
sellers["lookup_state"] = sellers["seller_zip_code_prefix"].apply(find_state_lookup)
print(sellers)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 47, Finished, Available, Finished)

                             seller_id  seller_zip_code_prefix  \
0     3442f8959a84dea7ee197c632cb2df15                   13023   
1     d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2     ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3     c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4     51a04a8a6bdcb23deccc82b0b80742cf                   12914   
...                                ...                     ...   
3090  98dddbc4601dd4443ca174359b237166                   87111   
3091  f8201cab383e484733266d1906e2fdfa                   88137   
3092  74871d19219c7d518d0090283e03c137                    4650   
3093  e603cf3fec55f8697c9059638d6c8eb5                   96080   
3094  9e25199f6ef7e7c347120ff175652c3b                   12051   

            seller_city seller_state seller_zip_code_prefix_str  \
0              campinas           SP                      13023   
1            mogi guacu           SP                      13844   
2     

In [48]:
#Drop the unnecessary columns

drop_columns = ["seller_zip_code_prefix", "seller_city", "seller_state", "matched_city", "matched_state", "validate_city_match", "validate_state_match"]
sellers_cleaned = sellers.drop(drop_columns, axis=1)
print(sellers_cleaned.head())

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 49, Finished, Available, Finished)

                          seller_id seller_zip_code_prefix_str  \
0  3442f8959a84dea7ee197c632cb2df15                      13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                      13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                      20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                      04195   
4  51a04a8a6bdcb23deccc82b0b80742cf                      12914   

         lookup_city lookup_state  
0           Campinas           SP  
1         Mogi Guaçu           SP  
2     Rio de Janeiro           RJ  
3          São Paulo           SP  
4  Bragança Paulista           SP  


In [49]:
# Replace column names
col_names = ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]
sellers_cleaned.columns = col_names
print(sellers_cleaned.head())

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 50, Finished, Available, Finished)

                          seller_id seller_zip_code_prefix        seller_city  \
0  3442f8959a84dea7ee197c632cb2df15                  13023           Campinas   
1  d1b65fc7debc3361ea86b5f14c68d2e2                  13844         Mogi Guaçu   
2  ce3ad9de960102d0677a81f5d0bb7b2d                  20031     Rio de Janeiro   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                  04195          São Paulo   
4  51a04a8a6bdcb23deccc82b0b80742cf                  12914  Bragança Paulista   

  seller_state  
0           SP  
1           SP  
2           RJ  
3           SP  
4           SP  


In [50]:
# Convert all text columns to pandas string dtype
sellers_cleaned = sellers_cleaned.astype("string", copy=True, errors='raise')

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 51, Finished, Available, Finished)

In [51]:
assert all(sellers_cleaned.dtypes == "string")

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 52, Finished, Available, Finished)

In [52]:
# Write the table to the silver lakehouse as a delta table
# Convert pandas to Spark
spark_sellers = spark.createDataFrame(sellers_cleaned)

# Save as a Delta table in Silver Lakehouse
silver_path = "SilverLakehouse.dbo.olist_sellers_cleaned"
spark_sellers.write.format("delta").mode("overwrite").saveAsTable(silver_path)

StatementMeta(, bcd65010-798b-44ba-b71a-48854c635e5e, 53, Finished, Available, Finished)